In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
volume_path= "/Volumes/main/default/raw_data/trips.json"
target_table="main.default.uber_trips_curated"
raw_schema = StructType([
    StructField("event_id", StringType(), nullable=False),
    StructField("trip_id", StringType(), nullable=False),
    StructField("city", StringType(), nullable=True),
    StructField("driver_id", StringType(), nullable=True),
    StructField("rider_id", StringType(), nullable=True),
    StructField("status", StringType(), nullable=True),
    StructField("fare_inr", DoubleType(), nullable=True),
    StructField("surge_multiplier", DoubleType(), nullable=True),
    StructField("distance_km", DoubleType(), nullable=True),
    StructField("event_ts", StringType(), nullable=True),
    StructField("updated_at", StringType(), nullable=True),
])

In [0]:
df1=spark.read.format("json").schema(raw_schema).load(volume_path)


In [0]:
df1.show()

In [0]:

df_cleaned=(df1
    .withColumn("city",F.lower(F.trim(F.col("city"))))
    .withColumn("status",F.trim(F.col("status")))
    .withColumn("fare_inr",F.round(F.col("fare_inr"),2))
    .withColumn("event_ts",F.try_to_timestamp(F.col("event_ts")))
    .withColumn("updated_at",F.to_timestamp(F.col("updated_at")))
    .withColumn("trip_date",F.date_format(F.col("event_ts"),"yyyy-MM-dd"))
    .withColumn("surge_multiplier",F.round(F.col("surge_multiplier"),2))
    .withColumn("distance_km",F.round(F.col("distance_km"),2)))

df_cleaned.show()

In [0]:
df_valid = df_cleaned.filter(
    (F.col("trip_id").isNotNull()) &
    (F.col("event_ts").isNotNull()) &
    (F.col("status") == "completed") &
    (F.coalesce(F.col("fare_inr"), F.lit(0.0)) > 0)
)

In [0]:
df_valid.show()

In [0]:
dedup_window = Window.partitionBy("trip_id").orderBy(F.col("updated_at").desc())

In [0]:
df_valid.withColumn("rank", F.row_number().over(dedup_window)).filter(F.col("rank") == 1).drop("rank")

In [0]:
df_valid.show()


# 4. Derived metrics + driver cumulative earnings window



In [0]:
driver_window= (Window.partitionBy("driver_id", "trip_date").orderBy(F.col("event_ts")).rowsBetween(Window.unboundedPreceding,Window.currentRow))

In [0]:
df_transformed2 = (
    df_valid
    .withColumn(
        "fare_per_km",
        F.when(F.col("distance_km") > 0, F.round(F.col("fare_inr") / F.col("distance_km"), 2))
        .otherwise(F.lit(0.0))
    )
    .withColumn(
        "surge_premium_inr",
        F.when(
            (F.col("status") == "COMPLETED") & (F.col("surge_multiplier") > 1.0),
            F.round(F.col("fare_inr") * (1.0 - (1.0 / F.col("surge_multiplier"))), 2)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "driver_daily_cum_fare_inr",
        F.round(
            F.sum(
                F.when(F.col("status") == "COMPLETED", F.col("fare_inr")).otherwise(F.lit(0.0))
            ).over(driver_window),
            2
        )
    )
    .withColumn("processed_at", F.current_timestamp())
)

In [0]:
df_transformed2.show()